<div style="text-align: center;">
  <span style="color: lightskyblue; font-weight: bold; font-size: 50px;">
    Test File for Depth Camera (D435i)
  </span>
</div>

<span style="color: magenta; font-weight: bold; font-size: 30px;">
Coordinate Information
</span>

<span style="color: white; font-weight: bold; font-size: 20px;">
- pixel: (u,v), origin = top-left point, u: left-to-right, v: top-to-bottom
</span>

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import rby1_lib_ysw as ysw
import logging
import sys
import time
import pyrealsense2 as rs
import cv2

np.set_printoptions(suppress=True, precision=5, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Check recognition of D435i
</span>

In [2]:
ctx = rs.context()
devices = ctx.query_devices()

print("Number of RealSense devices:", len(devices))

for dev in devices:
    print("Name:", dev.get_info(rs.camera_info.name))
    print("Serial:", dev.get_info(rs.camera_info.serial_number))
    print("Firmware:", dev.get_info(rs.camera_info.firmware_version))

Number of RealSense devices: 1
Name: Intel RealSense D435I
Serial: 250122070658
Firmware: 5.15.1.55


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization of Robot
</span>

In [2]:
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot
robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
# robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_ONLY)
if (robot != None):
    model = robot.model()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!
[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Reset the Control Manager!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************





<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Power OFF
</span>

In [26]:
ysw.powerOFF(robot, robotConfig)

****************************** [powerOFF Starts] *******************************

[INFO] SUCCESS: Disable Grippers!
[INFO] SUCCESS: Disable the Control Manager!

****************************** [PowerOFF is Done] ******************************





True

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization of Vision
</span>

In [2]:
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(640,480), fps=30)
d435i.initialize()

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Simple Test of D435i
</span>

In [3]:
# Simple test of RGB-D camera
# Click ESC: Terminate
d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()
        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)
        
        if (d435i.rgb_on):
            d435i.imshow("color_image", color_image)
        if (d435i.depth_on):
            d435i.imshow("depth_image", depth_colorMap)
        
        key = d435i.waitKey(1)
        if key == 27:       # ESC key
            break
        
finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()
        

[INFO] D435i pipeline started


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Distance of center pixel
</span>

In [ ]:
# Get distance value of center pixel of image
# Click ESC: Terminate
d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()
        
        if color_frame is None or depth_frame is None:
            continue
        
        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        h, w, _ = color_image.shape
        u = w // 2      # origin is left    
        v = h // 2      # origin is top

        distance = d435i.getDepth(depth_frame, (u, v))   # unit = [m]

        cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,                   
            f"Depth: {distance:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
3D position of center pixel based on Camera Reference Frame
</span>

In [4]:
# Show center 3D point position [x,y,z] in camera frame
d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        h, w, _ = color_image.shape
        u = w // 2      # origin is left    
        v = h // 2      # origin is top 

        pos = d435i.get3DPos(depth_frame, (u,v))
        X, Y, Z = pos

        cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,
            f"X:{X:.3f}, Y:{Y:.3f}, Z:{Z:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
3D position of clicked point 
</span>

In [3]:
d435i.openWindowWithMouse("color_image")

d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        if d435i.clicked_point is not None:
            u, v = d435i.clicked_point

            X, Y, Z = d435i.get3DPos(depth_frame, (u,v))

            cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
            cv2.putText(
                color_image,
                f"X:{X:.3f}, Y:{Y:.3f}, Z:{Z:.3f} m",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            print(f"Pixel=({u}, {v}), Camera 3D=({X:.3f}, {Y:.3f}, {Z:.3f}) m")

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

[INFO] D435i pipeline started
Clicked pixel: u=209, v=285
Pixel=(209, 285), Camera 3D=(-0.641, 0.248, 3.898) m
Pixel=(209, 285), Camera 3D=(-0.630, 0.243, 3.826) m
Pixel=(209, 285), Camera 3D=(-0.641, 0.248, 3.898) m
Pixel=(209, 285), Camera 3D=(-0.633, 0.245, 3.850) m
Pixel=(209, 285), Camera 3D=(-0.618, 0.239, 3.756) m
Pixel=(209, 285), Camera 3D=(-0.633, 0.245, 3.850) m
Pixel=(209, 285), Camera 3D=(-0.622, 0.240, 3.779) m
Pixel=(209, 285), Camera 3D=(-0.637, 0.246, 3.874) m
Pixel=(209, 285), Camera 3D=(-0.641, 0.248, 3.898) m
Pixel=(209, 285), Camera 3D=(-0.630, 0.243, 3.826) m
Pixel=(209, 285), Camera 3D=(-0.641, 0.248, 3.898) m
Pixel=(209, 285), Camera 3D=(-0.633, 0.245, 3.850) m
Pixel=(209, 285), Camera 3D=(-0.633, 0.245, 3.850) m
Pixel=(209, 285), Camera 3D=(-0.633, 0.245, 3.850) m
Pixel=(209, 285), Camera 3D=(-0.626, 0.242, 3.802) m
Pixel=(209, 285), Camera 3D=(-0.630, 0.243, 3.826) m
Pixel=(209, 285), Camera 3D=(-0.630, 0.243, 3.826) m
Pixel=(209, 285), Camera 3D=(-0.622, 0.24

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Center 3D position of Colored Object
</span>

In [3]:
# 빨간색 HSV 범위
red_mask_list = [
    (np.array([0, 80, 80]), np.array([10, 255, 255])),
    (np.array([170, 80, 80]), np.array([180, 255, 255]))
]
min_area = 300

d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = d435i.detectColorObjectContour(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=red_mask_list,
            minimum_area=min_area
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"pixel={obj['pixel']}, "
                f"pos={np.round(obj['pos'], 3)}, "
                f"area={obj['area']:.1f}"
            )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)
        d435i.imshow("mask", mask)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

[object 0] pixel=[367, 407], pos=[0.043 0.119 0.449], area=364.5
[object 0] pixel=[381, 389], pos=[0.054 0.108 0.459], area=413.0
[object 0] pixel=[396, 409], pos=[0.162 0.302 1.128], area=434.0
[object 0] pixel=[397, 383], pos=[0.068 0.105 0.468], area=950.0
[object 0] pixel=[390, 353], pos=[0.063 0.083 0.475], area=841.0
[object 0] pixel=[346, 341], pos=[0.029 0.073 0.47 ], area=469.5
[object 0] pixel=[329, 370], pos=[0.015 0.096 0.47 ], area=954.5
[object 0] pixel=[396, 387], pos=[0.069 0.112 0.482], area=1437.0
[object 0] pixel=[379, 353], pos=[0.056 0.085 0.486], area=485.5
[object 0] pixel=[357, 348], pos=[0.038 0.081 0.482], area=620.0
[object 0] pixel=[351, 371], pos=[0.033 0.099 0.482], area=946.0


QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 